## Multilingual sentence embeddings and SVM evaluation

This notebook evaluates a frozen multilingual sentence encoder with linear support vector machines (SVMs) under two training conditions: English only (`EN_only`) and English plus auxiliary languages (`EN_plus_auxiliary`). Evaluation and threshold selection remain English-only. The workflow is exploratory and is not a controlled language-data ablation: sample size and label/domain composition change together.

**Execution:** Run cells in order. Attach the frozen NLP setup dataset to Kaggle. Set `EMBEDDING_MODE = "restore"` to reuse a matching saved embedding dataset, or `"extract"` to regenerate embeddings on a GPU with access to the pinned Hugging Face model revision. Classifier training and evaluations then run from the selected cache. The notebook does not automatically recover previously trained SVM bundles; saved training outputs must be restored separately or retrained.

**Provenance:** Displayed outputs are retained from the original Kaggle execution. They are historical evidence, not results of running this edited copy.

In [2]:
import torch

# Reuse of saved embeddings does not require GPU acceleration.
EMBEDDING_MODE = "restore"  # Select "extract" to generate a new embedding cache.

if EMBEDDING_MODE not in {"restore", "extract"}:
    raise ValueError("EMBEDDING_MODE must be 'restore' or 'extract'.")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    probe = torch.ones(4, device="cuda:0")
    assert (probe + probe).sum().item() == 8
elif EMBEDDING_MODE == "extract":
    raise RuntimeError("GPU acceleration is required for embedding extraction.")

print("ENVIRONMENT_OK — no model loaded.")

PyTorch: 2.11.0+cu130
CUDA disponibile: True
GPU: Tesla T4
GPU_OK — nessun modello caricato.


## 01 · Runtime configuration

The embedding extraction path requires a CUDA GPU. Cached embeddings can be evaluated without GPU acceleration.

In [3]:
from pathlib import Path
from collections import Counter
import importlib.util
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

MULTILINGUAL_CONFIG = {
    'experiment': 'multilingual_encoder_SVM_EN_vs_MULTI_v1',
    'development_status': 'exploratory; follows prior outer analyses',
    'split_version': 'v2', 'taxonomy_version': 'project_v1_96fine',
    'training_seed': 42, 'folds': [1, 2, 3],
    'model_id': 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    'revision': '4328cf26390c98c5e3c738b4460a05b95f4911f5',
    'backbone': 'XLMRobertaModel', 'embedding_dimension': 768,
    'encoder_frozen': True, 'encoder_mode': 'eval',
    'encoder_precision': 'float32', 'attention': 'eager',
    'extraction_batch_size': 2,
    'max_length': 128, 'length_includes_special_tokens': True,
    'chunk_overlap_tokens': 0, 'chunk_policy': 'all content tokens, contiguous tokenizer chunks, no document truncation',
    'pooling': 'attention-mask mean pooling including special tokens',
    'chunk_normalization': 'L2',
    'document_aggregation': 'content-token-weighted mean of normalized chunks, then L2',
    'aggregation_note': 'project-defined document representation, not the default single-sequence model pipeline',
    'feature_standardization': None,
    'training_conditions': {
        'EN_only': ['EN'],
        'EN_plus_auxiliary': ['EN', 'BG', 'HI', 'PT', 'RU']
    },
    'validation_language': 'EN', 'outer_language': 'EN',
    'auxiliary_filter': 'existing make_fold grouping and exclusions; no new split',
    'classifier': {'C': 1.0, 'penalty': 'l2', 'loss': 'squared_hinge',
        'dual': 'auto', 'tol': 0.0001, 'max_iter': 10000,
        'class_weight': None, 'random_state': 42},
    'threshold_grid': [-1.0, -.9, -.8, -.7, -.6, -.5, -.4, -.3, -.2, -.1, 0., .1, .2, .3, .5],
    'selection': 'sample-F1 on inner validation, separately per level, fold and training condition',
    'threshold_tie_break': 'larger threshold', 'fixed_threshold_comparator': 0.0,
    'decoder': 'strictly greater than threshold; argmax fallback separately per level',
    'constant_targets': 'all-negative -> negative infinity; all-positive -> positive infinity',
    'hierarchy_postprocessing': False, 'Other_exclusive': False,
    'refit_train_plus_validation': False,
    'primary_metric': 'sample_F1_fine_project_96',
    'planned_systems': 2, 'planned_classifier_fits': 6,
    'labels_used_during_embedding_extraction': False,
    'course_test_used_for_modeling': False,
    'limitations': [
        'More auxiliary documents change sample size and label/domain distributions together.',
        'Fixed C preserves the solver recipe, not identical effective regularization as sample size changes.',
        'Lexical duplicate audit does not rule out translations or all paraphrases.',
        'Comparison with prior English MPNet also changes encoder and chunk length; it is not a pure data ablation.',
        'One training seed and exploratory outer evaluation.'
    ]
}

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/nlp_setup')
MULTI_ROOT = OUTPUT_ROOT / 'multilingual'
MULTI_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT = INPUT_ROOT / 'datasets/nizarmabrouk/nlp-semeval-2025-dati-e-setup'
KIT_DIR = DATA_ROOT / 'kit_setup_nlp'
if not (KIT_DIR / 'nlp_setup.py').is_file():
    raise FileNotFoundError('Attach the NLP SemEval 2025 - Data and Setup dataset to this notebook.')

config_serialized = json.dumps(MULTILINGUAL_CONFIG, ensure_ascii=False, sort_keys=True, indent=2)
config_sha256 = hashlib.sha256(config_serialized.encode('utf-8')).hexdigest()
assert config_sha256 == 'cc51cbca204a88dde5397f2c9cf975ac4b8899e96e693d8f480d6dd0d476f522', 'Configuration differs from the frozen experiment specification.'
(MULTI_ROOT / ('experiment_config_' + config_sha256[:12] + '.json')).write_text(config_serialized, encoding='utf-8')

spec = importlib.util.spec_from_file_location('nlp_setup_notebook07', KIT_DIR / 'nlp_setup.py')
setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(setup)
taxonomy = json.loads((KIT_DIR / 'tassonomia_progetto_v1.json').read_text(encoding='utf-8'))
expected = json.loads((KIT_DIR / 'expected_data_fingerprints.json').read_text(encoding='utf-8'))
print('Loading data and verifying fingerprints...', flush=True)
training_source = setup.locate_source(DATA_ROOT, 'training_NLP.zip', 'training')
test_source = setup.locate_source(DATA_ROOT, 'test_NLP.zip', 'test')
try:
    train_records = setup.load_training(training_source, taxonomy)
    course_texts = setup.load_course_texts(test_source)
finally:
    for source in (training_source, test_source):
        if source.archive is not None:
            source.archive.close()
setup.require(setup.fingerprint_records(train_records) == expected['training_records'], 'Training data fingerprint mismatch.')
setup.require(setup.fingerprint_records(course_texts) == expected['course_texts'], 'Test text fingerprint mismatch.')
similarity = json.loads((KIT_DIR / 'near_duplicate_audit_reference.json').read_text(encoding='utf-8'))
manifest = setup.validate_manifest(train_records, similarity, KIT_DIR / 'split_manifest_v2.csv')
assert len(similarity['excluded_training_keys']) == 11
assert len(taxonomy['coarse']) == 22 and len(taxonomy['fine']) == 96

fold_rows, needed_keys = [], set()
for fold in (1, 2, 3):
    en = setup.make_fold(train_records, manifest, similarity, fold, False)
    multi = setup.make_fold(train_records, manifest, similarity, fold, True)
    for part in ('inner_validation', 'outer_evaluation'):
        assert [r['key'] for r in en[part]] == [r['key'] for r in multi[part]]
        assert all(r['language'] == 'EN' for r in multi[part])
    train_keys = {r['key'] for r in multi['inner_train']}
    held_keys = {r['key'] for part in ('inner_validation', 'outer_evaluation') for r in multi[part]}
    assert not train_keys & held_keys
    assert not train_keys & set(similarity['excluded_training_keys'])
    for members in similarity['groups'].values():
        assert not (train_keys.intersection(members) and held_keys.intersection(members))
    for part in ('inner_train', 'inner_validation', 'outer_evaluation'):
        needed_keys.update(r['key'] for r in multi[part])
    fold_rows.append({'fold': fold, 'EN_train': len(en['inner_train']),
                     'multilingual_train': len(multi['inner_train']),
                     'EN_validation': len(en['inner_validation']),
                     'EN_outer': len(en['outer_evaluation'])})
assert [r['multilingual_train'] for r in fold_rows] == [1502, 1501, 1503]
assert [r['EN_train'] for r in fold_rows] == [213, 212, 214]

# Embeddings are generated without labels using a frozen encoder.
# Fold-specific training partitions are applied only during classifier fitting.
embedding_records = [
    {key: record[key] for key in ('key', 'document_id', 'language', 'text', 'raw_sha256')}
    for record in sorted(train_records, key=lambda r: r['key']) if record['key'] in needed_keys
]
assert len(embedding_records) == len(needed_keys) == 1688
assert all(r['key'].startswith('train/') for r in embedding_records)
assert not needed_keys.intersection(similarity['excluded_training_keys'])
language_counts = dict(Counter(r['language'] for r in embedding_records))
assert language_counts == {'BG': 400, 'EN': 399, 'HI': 359, 'PT': 397, 'RU': 133}
manifest.to_csv(MULTI_ROOT / 'verified_split_manifest_v2.csv', index=False)
setup.save_json(MULTI_ROOT / 'input_fingerprints.json', expected)
setup.save_json(MULTI_ROOT / 'taxonomy.json', taxonomy)
setup.save_json(MULTI_ROOT / 'embedding_document_index.json', [
    {key: r[key] for key in ('key', 'document_id', 'language', 'raw_sha256')}
    for r in embedding_records
])
multilingual_setup_result = {
    'status': 'MULTILINGUAL_SETUP_OK', 'config_sha256': config_sha256,
    'training_original': len(train_records), 'documents_for_embeddings': len(embedding_records),
    'languages': language_counts, 'folds': fold_rows,
    'similarity_audit': 'frozen reference reused after exact data fingerprint checks',
    'course_test_texts_checked_for_data_identity_only': len(course_texts),
    'course_test_in_embedding_cache': False, 'course_test_labels_loaded': False,
    'model_weights_loaded': False, 'embedding_extraction_started': False,
    'training_started': False, 'environment': setup.environment_info(),
    'output_directory': str(MULTI_ROOT)
}
setup.save_json(MULTI_ROOT / 'notebook07_setup_summary.json', multilingual_setup_result)
display(pd.DataFrame(fold_rows))
print(json.dumps(multilingual_setup_result, ensure_ascii=False, indent=2))


Caricamento dei dati e verifica delle impronte...


,fold,EN_train,multilingual_train,EN_validation,EN_outer
0,1,213,1502,53,133
1,2,212,1501,53,134
2,3,214,1503,53,132


{
  "status": "MULTILINGUAL_SETUP_OK",
  "config_sha256": "cc51cbca204a88dde5397f2c9cf975ac4b8899e96e693d8f480d6dd0d476f522",
  "training_original": 1699,
  "documents_for_embeddings": 1688,
  "languages": {
    "BG": 400,
    "EN": 399,
    "HI": 359,
    "PT": 397,
    "RU": 133
  },
  "folds": [
    {
      "fold": 1,
      "EN_train": 213,
      "multilingual_train": 1502,
      "EN_validation": 53,
      "EN_outer": 133
    },
    {
      "fold": 2,
      "EN_train": 212,
      "multilingual_train": 1501,
      "EN_validation": 53,
      "EN_outer": 134
    },
    {
      "fold": 3,
      "EN_train": 214,
      "multilingual_train": 1503,
      "EN_validation": 53,
      "EN_outer": 132
    }
  ],
  "similarity_audit": "frozen reference reused after exact data fingerprint checks",
  "course_test_texts_checked_for_data_identity_only": 178,
  "course_test_in_embedding_cache": false,
  "course_test_labels_loaded": false,
  "model_weights_loaded": false,
  "embedding_extraction_starte

## 02 · Frozen configuration and split verification

Verify the source-data fingerprints, 22 coarse and 96 fine labels, near-duplicate exclusions, and three frozen folds. Document embeddings are extracted without labels; SVM splits are applied afterward.

In [ ]:
import gc
import hashlib
import json
import time
import uuid
from collections import Counter
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import hf_hub_download

def sem_pool(hidden, mask):
    weights = mask.unsqueeze(-1).to(hidden.dtype)
    pooled = (hidden * weights).sum(1) / weights.sum(1).clamp_min(1.)
    return F.normalize(pooled, p=2, dim=1)

def sem_aggregate(vectors, counts):
    counts = np.asarray(counts, dtype=np.float64)
    assert (counts > 0).all()
    pooled = np.average(vectors.astype(np.float64), axis=0, weights=counts)
    norm = np.linalg.norm(pooled)
    assert np.isfinite(norm) and norm > 1e-12
    return (pooled / norm).astype(np.float32)

def multilingual_extract_impl():
    missing = [name for name in ('setup', 'embedding_records', 'manifest', 'MULTILINGUAL_CONFIG', 'multilingual_setup_result', 'expected', 'taxonomy') if name not in globals()]
    if missing:
        raise RuntimeError(f'Run the data setup section before extraction: {missing}')
    assert torch.cuda.is_available(), 'GPU acceleration is required for embedding extraction.'
    setup.seed_everything(42)
    cfg = json.loads(json.dumps(MULTILINGUAL_CONFIG))
    config_digest = hashlib.sha256(json.dumps(cfg, ensure_ascii=False, sort_keys=True, indent=2).encode('utf-8')).hexdigest()
    assert config_digest == 'cc51cbca204a88dde5397f2c9cf975ac4b8899e96e693d8f480d6dd0d476f522'
    assert multilingual_setup_result['status'] == 'MULTILINGUAL_SETUP_OK'
    # Resolve the pinned model revision without substituting the current main branch.
    dest = Path('/kaggle/working/nlp_setup/multilingual') / ('multilingual_embeddings_' + uuid.uuid4().hex[:10])
    dest.mkdir(parents=True, exist_ok=False)
    def save(name, obj):
        (dest / name).write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')
    save('config.json', cfg)
    save('environment.json', setup.environment_info())
    manifest.to_csv(dest / 'split_manifest.csv', index=False)
    records = sorted(embedding_records, key=lambda r: r['key'])
    assert len(records) == 1688 and len({r['key'] for r in records}) == 1688
    assert all('fine' not in r and 'coarse' not in r for r in records)
    save('input_fingerprints.json', expected)
    save('taxonomy.json', taxonomy)
    save('status.json', {'state': 'extracting', 'training_started': False})
    # The encoder receives document identifiers and texts, not target labels.
    save('documents.json', [{'id': r['document_id'], 'key': r['key'],
        'language': r['language'], 'raw_sha256': r['raw_sha256'],
        'text_sha256': hashlib.sha256(r['text'].encode('utf-8')).hexdigest()} for r in records])
    print('Output directory:', dest, flush=True)
    print('Loading tokenizer at revision:', cfg['revision'], flush=True)
    sentence_config = json.loads(Path(hf_hub_download(cfg['model_id'], 'sentence_bert_config.json', revision=cfg['revision'])).read_text())
    pooling_config = json.loads(Path(hf_hub_download(cfg['model_id'], '1_Pooling/config.json', revision=cfg['revision'])).read_text())
    assert sentence_config['max_seq_length'] == cfg['max_length']
    assert pooling_config['pooling_mode_mean_tokens'] is True
    assert pooling_config['word_embedding_dimension'] == 768
    assert not any(value for key, value in pooling_config.items() if key.startswith('pooling_mode_') and key != 'pooling_mode_mean_tokens')
    save('source_sentence_config.json', sentence_config)
    save('source_pooling_config.json', pooling_config)
    tok = AutoTokenizer.from_pretrained(cfg['model_id'], revision=cfg['revision'],
                                      use_fast=True, trust_remote_code=False)
    assert tok.is_fast, 'A fast tokenizer is required for verifiable chunking.'
    tok.truncation_side = 'right'
    tok.padding_side = 'right'
    tok.save_pretrained(dest / 'tokenizer')
    special = tok.num_special_tokens_to_add(pair=False)
    assert 0 < special < cfg['max_length']
    all_ids, all_masks, ownership, counts, lengths = [], [], [], [], []
    first_indices = []
    for i, record in enumerate(records):
        full_ids = tok(record['text'], add_special_tokens=False, truncation=False,
                       return_attention_mask=False, return_token_type_ids=False, verbose=False)['input_ids']
        assert len(full_ids) > 0, 'Document contains no content tokens.'
        encoded = tok(record['text'], max_length=cfg['max_length'], truncation=True,
                      stride=0, return_overflowing_tokens=True, padding='max_length',
                      return_token_type_ids=False, return_tensors='pt')
        n = len(encoded['input_ids'])
        chunk_counts = (encoded['attention_mask'].sum(1) - special).tolist()
        assert sum(chunk_counts) == len(full_ids), 'Tokens were lost or duplicated during chunking.'
        # Reconstruct content-token order, excluding inserted special tokens.
        rebuilt = []
        for ids, mask in zip(encoded['input_ids'].tolist(), encoded['attention_mask'].tolist()):
            active = ids[:sum(mask)]
            # XLM-R single-sequence encoding uses one start token and one end token.
            assert special == 2 and active[0] == tok.cls_token_id and active[-1] == tok.sep_token_id
            rebuilt.extend(active[1:-1])
        assert rebuilt == full_ids, 'Content-token order was not preserved.'
        truncated = tok(record['text'], max_length=cfg['max_length'], truncation=True,
                        padding='max_length', return_token_type_ids=False)['input_ids']
        assert truncated == encoded['input_ids'][0].tolist()
        first_indices.append(len(ownership))
        all_ids.append(encoded['input_ids'])
        all_masks.append(encoded['attention_mask'])
        ownership.extend([i] * n)
        counts.extend(chunk_counts)
        lengths.append({'id': record['document_id'], 'key': record['key'], 'language': record['language'], 'content_tokens': len(full_ids), 'chunks': n})
        if (i + 1) % 200 == 0 or i + 1 == len(records):
            print(f'Tokenized {i + 1}/{len(records)} documents', flush=True)
    inputs = torch.cat(all_ids)
    masks = torch.cat(all_masks)
    save('document_lengths.json', lengths)
    print(f'{len(records)} documents, {len(inputs)} chunks. Loading encoder.', flush=True)
    encoder = AutoModel.from_pretrained(cfg['model_id'], revision=cfg['revision'],
                                       trust_remote_code=False, attn_implementation=cfg['attention']).float().to('cuda:0')
    assert encoder.config.model_type == 'xlm-roberta' and encoder.config.hidden_size == 768
    assert encoder.dtype == torch.float32
    encoder.requires_grad_(False)
    encoder.eval()
    vectors = []
    started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    with torch.inference_mode():
        for start in range(0, len(inputs), cfg['extraction_batch_size']):
            stop = start + cfg['extraction_batch_size']
            mask = masks[start:stop].to('cuda:0')
            hidden = encoder(input_ids=inputs[start:stop].to('cuda:0'), attention_mask=mask).last_hidden_state
            vectors.append(sem_pool(hidden, mask).cpu().numpy())
            if start % 100 == 0 or stop >= len(inputs):
                print(f'Processed chunks: {min(stop, len(inputs))}/{len(inputs)}', flush=True)
            del hidden, mask
    chunk_vectors = np.concatenate(vectors)
    owners = np.array(ownership)
    token_counts = np.array(counts)
    first = chunk_vectors[first_indices]
    whole = np.stack([sem_aggregate(chunk_vectors[owners == i], token_counts[owners == i]) for i in range(len(records))])
    for array in (first, whole):
        assert array.shape == (1688, 768) and np.isfinite(array).all()
        assert np.allclose(np.linalg.norm(array, axis=1), 1., atol=1e-5)
    single = np.array([row['chunks'] == 1 for row in lengths])
    assert np.allclose(first[single], whole[single], atol=1e-6)
    np.savez_compressed(dest / 'embeddings.npz', ids=np.array([r['document_id'] for r in records]),
                        keys=np.array([r['key'] for r in records]),
                        languages=np.array([r['language'] for r in records]), all_chunks=whole)
    np.savez_compressed(dest / 'chunk_embeddings.npz', vectors=chunk_vectors, document_index=owners, content_token_counts=token_counts)
    summary = {'status': 'MULTILINGUAL_EMBEDDINGS_OK', 'documents': 1688,
        'config_sha256': config_digest, 'languages': dict(Counter(r['language'] for r in records)),
        'model_id': cfg['model_id'], 'revision': cfg['revision'], 'embedding_dimension': 768,
        'max_length': cfg['max_length'], 'content_capacity': cfg['max_length'] - special,
        'total_chunks': len(owners), 'documents_with_multiple_chunks': int((~single).sum()),
        'mean_chunks_per_document': float(len(owners) / 1688),
        'encoder_seconds': time.perf_counter() - started,
        'peak_allocated_GiB': torch.cuda.max_memory_allocated() / 1024**3,
        'labels_used': False, 'training_started': False, 'evaluation_performed': False,
        'output_directory': str(dest)}
    summary['model_parameters_updated'] = False
    summary['course_test_in_cache'] = False
    save('summary.json', summary)
    save('status.json', {'state': 'complete', 'training_started': False})
    hashes = {p.relative_to(dest).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
              for p in dest.rglob('*') if p.is_file()}
    save('files_sha256.json', hashes)
    with np.load(dest / 'embeddings.npz', allow_pickle=False) as saved:
        assert np.array_equal(saved['all_chunks'], whole)
        assert len(set(saved['keys'].tolist())) == 1688
    del encoder
    archive = dest.with_suffix('.zip')
    with ZipFile(archive, 'w', ZIP_DEFLATED) as z:
        for file in sorted(dest.rglob('*')):
            if file.is_file():
                z.write(file, file.relative_to(dest))
    with ZipFile(archive) as z:
        assert z.testzip() is None
    print(json.dumps(summary, indent=2))
    print('Embedding archive:', archive)
    return summary

def extract_multilingual_embeddings():
    try:
        return multilingual_extract_impl()
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if EMBEDDING_MODE == 'extract':
    multilingual_embedding_result = extract_multilingual_embeddings()


## 03 · Embedding generation (optional)

The pinned multilingual MPNet encoder uses tokenizer overflow chunks (128 tokens including special tokens), mean pooling, L2 normalization, and content-token-weighted aggregation. This section executes only in `extract` mode. The original notebook did not save an output for this extraction step.

In [5]:
if EMBEDDING_MODE == 'restore':
    import json
    from pathlib import Path
    from shutil import unpack_archive

    import numpy as np

    archive_stem = 'multilingual_embeddings_be6df64d78'
    input_root = Path('/kaggle/input')
    expected_config_digest = config_sha256

    def find_embedding_cache(root):
        for file in sorted(root.rglob('embeddings.npz')):
            summary_path = file.parent / 'summary.json'
            if summary_path.is_file():
                summary = json.loads(summary_path.read_text(encoding='utf-8'))
                if (summary.get('status') == "MULTILINGUAL_EMBEDDINGS_OK"
                        and summary.get('config_sha256') == expected_config_digest):
                    return file.parent
        return None

    cache_directory = find_embedding_cache(input_root)

    if cache_directory is None:
        archive_path = next(input_root.rglob(archive_stem + '.zip'), None)
        if archive_path is None:
            raise FileNotFoundError(
                f'No matching embedding cache or {archive_stem}.zip found in /kaggle/input. '
                'Attach the embedding dataset before running this section.'
            )
        destination = Path('/kaggle/working') / archive_stem
        unpack_archive(str(archive_path), str(destination))
        cache_directory = find_embedding_cache(destination)

    if cache_directory is None:
        raise RuntimeError("Embedding summary does not match the frozen experiment configuration.")

    with np.load(cache_directory / 'embeddings.npz', allow_pickle=False) as cache:
        shape = cache['all_chunks'].shape
        assert shape == (1688, 768), f'Unexpected embedding shape: {shape}'
        assert len(set(cache["keys"].tolist())) == 1688, "Unexpected document keys"

    multilingual_embedding_result = {'output_directory': str(cache_directory)}
    print('EMBEDDINGS_RESTORED', shape)
    print('Cache directory:', cache_directory)
else:
    print('Using the newly extracted embedding cache:', multilingual_embedding_result['output_directory'])

EMBEDDINGS_RIPRISTINATI (1688, 768)
Percorso: /kaggle/input/datasets/nizarmabrouk/nlp-embeddings-multilingue


## 04 · Restore or select embedding cache

In `restore` mode, locate a matching cache or the original named archive under `/kaggle/input`. Cache integrity, fingerprints, taxonomy, and document identities are further checked before training. In `extract` mode, use the cache produced above. An existing cache is required; no silent extraction fallback is performed.

In [6]:
import hashlib
import json
import shutil
import time
import uuid
import warnings
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.svm import LinearSVC
from sklearn.exceptions import ConvergenceWarning

def semtrain_fit(x, y, params):
    models = []
    for j in range(y.shape[1]):
        unique = np.unique(y[:, j])
        if len(unique) == 1:
            models.append(float(unique[0]))
        else:
            model = LinearSVC(**params)
            with warnings.catch_warnings():
                warnings.simplefilter('error', ConvergenceWarning)
                model.fit(x, y[:, j])
            models.append(model)
    return models

def semtrain_scores(models, x):
    columns = []
    for model in models:
        if isinstance(model, float):
            assert model in (0., 1.)
            values = np.full(len(x), np.inf if model else -np.inf)
        else:
            assert list(model.classes_) == [0, 1]
            values = model.decision_function(x)
            assert np.isfinite(values).all()
        columns.append(values)
    scores = np.column_stack(columns)
    assert not np.isnan(scores).any() and not np.isneginf(scores).all(axis=1).any()
    return scores

def semtrain_decode(scores, threshold):
    p = scores > threshold
    empty = np.flatnonzero(~p.any(1))
    p[empty, scores[empty].argmax(1)] = True
    return p

def semtrain_f1(y, p):
    denominator = y.sum(1) + p.sum(1)
    return float(np.divide(2*(y*p).sum(1), denominator,
        out=np.zeros(len(y), dtype=float), where=denominator != 0).mean())

def semtrain_select(y, scores, grid):
    search = [{'threshold': t, 'validation_sample_f1': semtrain_f1(y, semtrain_decode(scores, t))} for t in grid]
    return max(search, key=lambda r: (r['validation_sample_f1'], r['threshold'])), search


def train_multilingual_conditions():
    required = ('setup', 'train_records', 'manifest', 'similarity', 'taxonomy', 'MULTILINGUAL_CONFIG')
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError('Run the data setup section first. Missing: ' + repr(missing))
    source = Path(multilingual_embedding_result['output_directory']) if 'multilingual_embedding_result' in globals() else Path('/kaggle/working/nlp_setup/multilingual/multilingual_embeddings_be6df64d78')
    needed = ('config.json', 'summary.json', 'documents.json', 'embeddings.npz',
              'split_manifest.csv', 'taxonomy.json', 'input_fingerprints.json', 'files_sha256.json')
    absent = [name for name in needed if not (source / name).is_file()]
    if absent:
        raise FileNotFoundError('Restore the saved embedding archive. Missing files: ' + repr(absent))
    read = lambda name: json.loads((source / name).read_text(encoding='utf-8'))
    cfg = read('config.json')
    assert cfg == MULTILINGUAL_CONFIG
    digest = hashlib.sha256(json.dumps(cfg, ensure_ascii=False, sort_keys=True, indent=2).encode()).hexdigest()
    assert digest == 'cc51cbca204a88dde5397f2c9cf975ac4b8899e96e693d8f480d6dd0d476f522'
    assert read('summary.json')['status'] == 'MULTILINGUAL_EMBEDDINGS_OK'
    hashes = read('files_sha256.json')
    for filename in needed:
        if filename != 'files_sha256.json':
            assert hashlib.sha256((source / filename).read_bytes()).hexdigest() == hashes[filename], 'Checksum mismatch: ' + filename
    assert read('taxonomy.json') == taxonomy
    assert read('input_fingerprints.json')['training_records'] == setup.fingerprint_records(train_records)
    pd.testing.assert_frame_equal(pd.read_csv(source / 'split_manifest.csv').reset_index(drop=True), manifest.reset_index(drop=True))
    documents = read('documents.json')
    lookup = {r['key']: r for r in train_records}
    for doc in documents:
        r = lookup[doc['key']]
        assert doc['id'] == r['document_id'] and doc['language'] == r['language']
        assert doc['raw_sha256'] == r['raw_sha256']
        assert doc['text_sha256'] == hashlib.sha256(r['text'].encode('utf-8')).hexdigest()
    with np.load(source / 'embeddings.npz', allow_pickle=False) as data:
        keys = data['keys'].astype(str).tolist()
        assert keys == [r['key'] for r in documents] and len(keys) == len(set(keys)) == 1688
        assert data['ids'].astype(str).tolist() == [r['id'] for r in documents]
        assert data['languages'].astype(str).tolist() == [r['language'] for r in documents]
        features = data['all_chunks'].copy()
    assert features.shape == (1688, 768) and np.isfinite(features).all()
    assert np.allclose(np.linalg.norm(features, axis=1), 1., atol=1e-5)
    index = {key: i for i, key in enumerate(keys)}
    dest = Path('/kaggle/working/nlp_setup/runs') / ('multilingual_SVM_EN_vs_MULTI_seed42_' + uuid.uuid4().hex[:10])
    dest.mkdir(parents=True, exist_ok=False)
    cache = dest / 'embedding_cache'
    cache.mkdir()
    for filename in needed:
        shutil.copyfile(source / filename, cache / filename)
    setup.save_json(dest / 'config.json', cfg)
    setup.save_json(dest / 'taxonomy.json', taxonomy)
    setup.save_json(dest / 'environment.json', setup.environment_info())
    setup.save_json(dest / 'embedding_source.json', {'directory': str(source), 'config_sha256': digest,
        'embeddings_sha256': hashes['embeddings.npz'],
        'cache_note': 'Subset of source files; files_sha256.json describes the original full extraction archive.'})
    manifest.to_csv(dest / 'split_manifest_v2.csv', index=False)
    rows = []
    print('Verified embedding cache. Training run:', dest, flush=True)
    try:
        for fold in (1, 2, 3):
            for condition, multilingual in [('EN_only', False), ('EN_plus_auxiliary', True)]:
                parts = setup.make_fold(train_records, manifest, similarity, fold, multilingual)
                train, val = parts['inner_train'], parts['inner_validation']
                train_keys, val_keys = [r['key'] for r in train], [r['key'] for r in val]
                held_keys = {r['key'] for r in parts['outer_evaluation']}
                assert len(train_keys) == len(set(train_keys))
                assert not set(train_keys) & (set(val_keys) | held_keys)
                assert not set(val_keys) & held_keys
                assert all(r['language'] in cfg['training_conditions'][condition] for r in train)
                assert all(r['language'] == 'EN' for r in val)
                expected_n = ([1502, 1501, 1503] if multilingual else [213, 212, 214])[fold-1]
                assert len(train) == expected_n and len(val) == 53
                x_train = features[[index[k] for k in train_keys]].astype(np.float64)
                x_val = features[[index[k] for k in val_keys]].astype(np.float64)
                folder = dest / condition / f'fold_{fold}'
                folder.mkdir(parents=True)
                print(f'Fold {fold}/3 — {condition}: {len(train)} training documents, {len(val)} validation documents.', flush=True)
                models, thresholds, searches, scores, supports = {}, {}, {}, {}, {}
                start = time.perf_counter()
                for level in ('coarse', 'fine'):
                    y = setup.binary_labels([r[level] for r in train], taxonomy[level])
                    y_val = setup.binary_labels([r[level] for r in val], taxonomy[level])
                    supports[level] = y.sum(0)
                    models[level] = semtrain_fit(x_train, y, cfg['classifier'])
                    scores[level] = semtrain_scores(models[level], x_val)
                    best, searches[level] = semtrain_select(y_val, scores[level], cfg['threshold_grid'])
                    thresholds[level] = best['threshold']
                    p = semtrain_decode(scores[level], best['threshold'])
                    other = taxonomy[level].index('Other')
                    rows.append({'condition': condition, 'fold': fold, 'level': level,
                        'threshold': best['threshold'], 'validation_F1': best['validation_sample_f1'],
                        'train_documents': len(train), 'validation_documents': len(val),
                        'threshold_at_grid_edge': best['threshold'] in (min(cfg['threshold_grid']), max(cfg['threshold_grid'])),
                        'only_Other_documents': int(((p.sum(1) == 1) & p[:, other]).sum()),
                        'distinct_labels_predicted': int(p.any(0).sum())})
                bundle = {'models': models, 'thresholds': thresholds, 'classes': taxonomy,
                    'train_keys': train_keys, 'validation_keys': val_keys,
                    'condition': condition, 'fold': fold, 'config_sha256': digest,
                    'embedding_sha256': hashes['embeddings.npz']}
                joblib.dump(bundle, folder / 'svm_models.joblib')
                setup.save_json(folder / 'threshold_search.json', searches)
                setup.save_json(folder / 'timing.json', {'training_and_validation_seconds': time.perf_counter()-start})
                np.savez_compressed(folder / 'validation_decision_scores.npz', keys=np.array(val_keys),
                    ids=np.array([r['document_id'] for r in val]), **scores)
                np.savez_compressed(folder / 'training_label_support.npz', **supports)
                print('Completed; selected thresholds:', thresholds, flush=True)
        assert len(rows) == 12
    except Exception as exc:
        setup.save_json(dest / 'status.json', {'status': 'FAILED_TRAINING', 'error': repr(exc)})
        raise
    pd.DataFrame(rows).to_csv(dest / 'validation_selection.csv', index=False)
    summary = {'status': 'MULTILINGUAL_TRAINING_OK', 'conditions': ['EN_only', 'EN_plus_auxiliary'],
        'folds': 3, 'seed': 42, 'config_sha256': digest, 'encoder_updated': False,
        'outer_evaluation_performed': False, 'course_test_labels_loaded': False,
        'validation_selection': rows, 'run_directory': str(dest)}
    setup.save_json(dest / 'training_summary.json', summary)
    setup.save_json(dest / 'status.json', {'status': 'training_and_selection_complete', 'outer_evaluated': False})
    archive_path = dest.with_suffix('.zip')
    with ZipFile(archive_path, 'w', ZIP_DEFLATED) as archive:
        for path in sorted(dest.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(dest).as_posix())
    with ZipFile(archive_path) as archive:
        assert archive.testzip() is None
    display(pd.DataFrame(rows))
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print('Training archive:', archive_path)
    return summary


multilingual_training_result = train_multilingual_conditions()


Cache verificata. Esperimento: /kaggle/working/nlp_setup/runs/multilingual_SVM_EN_vs_MULTI_seed42_35e21ad56f
Fold 1/3 — EN_only: 213 training, 53 validazione.
Completato; soglie: {'coarse': -0.3, 'fine': -0.6}
Fold 1/3 — EN_plus_auxiliary: 1502 training, 53 validazione.
Completato; soglie: {'coarse': -0.2, 'fine': -0.5}
Fold 2/3 — EN_only: 212 training, 53 validazione.
Completato; soglie: {'coarse': -0.4, 'fine': 0.5}
Fold 2/3 — EN_plus_auxiliary: 1501 training, 53 validazione.
Completato; soglie: {'coarse': -0.6, 'fine': -0.4}
Fold 3/3 — EN_only: 214 training, 53 validazione.
Completato; soglie: {'coarse': 0.5, 'fine': -0.3}
Fold 3/3 — EN_plus_auxiliary: 1503 training, 53 validazione.
Completato; soglie: {'coarse': -0.1, 'fine': -0.5}


,condition,fold,level,threshold,validation_F1,train_documents,validation_documents,threshold_at_grid_edge,only_Other_documents,distinct_labels_predicted
0,EN_only,1,coarse,-0.3,0.599146,213,53,False,22,12
1,EN_only,1,fine,-0.6,0.500908,213,53,False,14,35
2,EN_plus_auxiliary,1,coarse,-0.2,0.592243,1502,53,False,13,16
3,EN_plus_auxiliary,1,fine,-0.5,0.463179,1502,53,False,11,40
4,EN_only,2,coarse,-0.4,0.562771,212,53,False,16,17
5,EN_only,2,fine,0.5,0.434007,212,53,True,29,12
6,EN_plus_auxiliary,2,coarse,-0.6,0.537233,1501,53,False,3,20
7,EN_plus_auxiliary,2,fine,-0.4,0.447799,1501,53,False,11,34
8,EN_only,3,coarse,0.5,0.548428,214,53,True,29,10
9,EN_only,3,fine,-0.3,0.471069,214,53,False,31,15


{
  "status": "MULTILINGUAL_TRAINING_OK",
  "conditions": [
    "EN_only",
    "EN_plus_auxiliary"
  ],
  "folds": 3,
  "seed": 42,
  "config_sha256": "cc51cbca204a88dde5397f2c9cf975ac4b8899e96e693d8f480d6dd0d476f522",
  "encoder_updated": false,
  "outer_evaluation_performed": false,
  "course_test_labels_loaded": false,
  "validation_selection": [
    {
      "condition": "EN_only",
      "fold": 1,
      "level": "coarse",
      "threshold": -0.3,
      "validation_F1": 0.5991464510332435,
      "train_documents": 213,
      "validation_documents": 53,
      "threshold_at_grid_edge": false,
      "only_Other_documents": 22,
      "distinct_labels_predicted": 12
    },
    {
      "condition": "EN_only",
      "fold": 1,
      "level": "fine",
      "threshold": -0.6,
      "validation_F1": 0.5009083160026556,
      "train_documents": 213,
      "validation_documents": 53,
      "threshold_at_grid_edge": false,
      "only_Other_documents": 14,
      "distinct_labels_predicted": 35
 

## 05 · SVM training and validation selection

Fit independent binary linear SVMs for coarse and fine targets under both conditions across three folds (six condition/fold fits, each containing label-wise classifiers). Select a threshold using English validation sample-F1, with the frozen tie-breaking rule. Save model bundles, decision scores, class support, and configuration metadata.

In [7]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

run = Path(multilingual_training_result["run_directory"])
conditions = ["EN_only", "EN_plus_auxiliary"]

with np.load(run / "embedding_cache" / "embeddings.npz", allow_pickle=False) as cache:
    index = {key: i for i, key in enumerate(cache["keys"].astype(str))}
    features = cache["all_chunks"]

outer_results = {}
fold_rows = []
saved = {}

for condition in conditions:
    keys_all = []
    gold = {"coarse": [], "fine": []}
    predictions = {"coarse": [], "fine": []}
    fold_ids = []

    for fold, expected_size in [(1, 133), (2, 134), (3, 132)]:
        parts = setup.make_fold(
            train_records, manifest, similarity, fold,
            condition == "EN_plus_auxiliary"
        )
        outer = sorted(parts["outer_evaluation"], key=lambda r: r["key"])
        keys = [r["key"] for r in outer]
        bundle = joblib.load(run / condition / f"fold_{fold}" / "svm_models.joblib")

        assert bundle["condition"] == condition and bundle["fold"] == fold
        assert bundle["classes"] == taxonomy
        assert len(keys) == len(set(keys)) == expected_size
        assert all(r["language"] == "EN" for r in outer)
        assert set(keys).isdisjoint(bundle["train_keys"])
        assert set(keys).isdisjoint(bundle["validation_keys"])

        x = features[[index[key] for key in keys]].astype(np.float64)
        keys_all.extend(keys)
        fold_ids.extend([fold] * len(keys))
        row = {"condition": condition, "fold": fold, "documents": len(keys)}

        for level in ("coarse", "fine"):
            y = setup.binary_labels([r[level] for r in outer], taxonomy[level])
            scores = semtrain_scores(bundle["models"][level], x)
            pred = semtrain_decode(scores, bundle["thresholds"][level])
            gold[level].append(y)
            predictions[level].append(pred)
            row[f"F1_{level}"] = semtrain_f1(y, pred)

        fold_rows.append(row)

    assert len(keys_all) == len(set(keys_all)) == 399
    if condition == "EN_only":
        saved["keys"] = np.array(keys_all)
        saved["fold"] = np.array(fold_ids)
    else:
        assert keys_all == saved["keys"].tolist()
        assert fold_ids == saved["fold"].tolist()

    outer_results[condition] = {}
    for level in ("coarse", "fine"):
        y = np.concatenate(gold[level])
        pred = np.concatenate(predictions[level])
        if condition == "EN_only":
            saved[f"gold_{level}"] = y
            saved[f"classes_{level}"] = np.array(taxonomy[level])
        else:
            assert np.array_equal(y, saved[f"gold_{level}"])

        saved[f"{condition}_{level}"] = pred
        outer_results[condition][f"F1_{level}"] = semtrain_f1(y, pred)

by_fold = pd.DataFrame(fold_rows)
overall = pd.DataFrame.from_dict(outer_results, orient="index")
overall.loc["MULTI_meno_EN"] = overall.loc["EN_plus_auxiliary"] - overall.loc["EN_only"]

output = run.parent / f"{run.name}_outer_evaluation.npz"
np.savez_compressed(output, **saved)

display(by_fold.round(4))
display(overall.round(4))
print('MULTILINGUAL_OUTER_OK — 399 documents per condition')
print('Saved outer predictions:', output)

,condition,fold,documents,F1_coarse,F1_fine
0,EN_only,1,133,0.5221,0.4055
1,EN_only,2,134,0.5632,0.4577
2,EN_only,3,132,0.5397,0.4588
3,EN_plus_auxiliary,1,133,0.5575,0.4363
4,EN_plus_auxiliary,2,134,0.5432,0.4833
5,EN_plus_auxiliary,3,132,0.4803,0.3774


,F1_coarse,F1_fine
EN_only,0.5417,0.4407
EN_plus_auxiliary,0.5272,0.4326
MULTI_meno_EN,-0.0145,-0.0081


MULTILINGUAL_OUTER_OK — 399 articoli per sistema
Predizioni da scaricare: /kaggle/working/nlp_setup/runs/multilingual_SVM_EN_vs_MULTI_seed42_35e21ad56f_outer_evaluation.npz


## 06 · English outer evaluation

Evaluate the saved models on 399 English outer-evaluation documents; preserve fold-level and pooled sample-F1 metrics for both label granularities.

In [8]:
print('Manifest columns:', manifest.columns.tolist())
display(manifest.head(3))

Colonne del manifest: ['fold', 'partition', 'language', 'document_id', 'leakage_group_id', 'raw_sha256']


,fold,partition,language,document_id,leakage_group_id,raw_sha256
0,1,inner_train,EN,EN_CC_100000.txt,977cfe6a2e224815385d0aaf3f8e054f486e82c4666a74...,94c103cff506f33da3c9fcaae0f4c56bba68131ef9cfcb...
1,1,inner_train,EN,EN_CC_100002.txt,90f493ef3e90fd0489a2836c2b38e0fcad31ddd6e50d2d...,8efb8f54275d5deba8f3ddc055e76049db955677fe6f3f...
2,1,inner_train,EN,EN_CC_100004.txt,072033631c70c4e599b6b51578e194aed360069072d4dc...,f681f71b61fd27e6f754a7aee9ce3ba0564a710aa0bf87...


## 07 · Split manifest inspection

Inspect the columns used to join outer-evaluation predictions to leakage-group identifiers.

In [9]:
from pathlib import Path
import numpy as np
import pandas as pd

run = Path(multilingual_training_result["run_directory"])
file_pred = run.parent / f"{run.name}_outer_evaluation.npz"

with np.load(file_pred, allow_pickle=False) as archive:
    pred = {name: archive[name] for name in archive.files}

# Link internal prediction keys to document identifiers used in the manifest.
record_by_key = {r["key"]: r for r in train_records if r["language"] == "EN"}
keys = pred["keys"].astype(str)
articles = pd.DataFrame({
    "fold": pred["fold"],
    "document_id": [record_by_key[k]["document_id"] for k in keys],
})

# Preserve prediction order when joining the manifest.
manifest_en = manifest.loc[manifest["language"] == "EN"].set_index(
    ["fold", "document_id"], verify_integrity=True
)
matched = manifest_en.reindex(pd.MultiIndex.from_frame(articles))
assert len(articles) == articles["document_id"].nunique() == 399
assert matched["leakage_group_id"].notna().all(), "Document missing from the manifest"
assert not matched["partition"].isin(["inner_train", "inner_validation"]).any()
group_idx, groups = pd.factorize(matched["leakage_group_id"], sort=True)
assert len(groups) == 395, f'Found {len(groups)} groups; expected 395'

def f1_per_document(gold, prediction):
    gold = gold.astype(bool)
    prediction = prediction.astype(bool)
    denominator = gold.sum(axis=1) + prediction.sum(axis=1)
    return np.divide(
        2 * (gold & prediction).sum(axis=1), denominator,
        out=np.zeros(len(gold), dtype=float), where=denominator > 0
    )

levels = ["coarse", "fine"]
differences = []
for level in levels:
    gold = pred[f"gold_{level}"]
    en = f1_per_document(gold, pred[f"EN_only_{level}"])
    multi = f1_per_document(gold, pred[f"EN_plus_auxiliary_{level}"])
    differences.append(multi - en)

delta = np.column_stack(differences)
observed = delta.mean(axis=0)
# Check against rounded outer-evaluation differences.
assert np.allclose(observed, [-0.0145, -0.0081], atol=0.000051)

group_sums = np.zeros((len(groups), len(levels)))
np.add.at(group_sums, group_idx, delta)
group_sizes = np.bincount(group_idx)

rng = np.random.default_rng(20261104)
bootstrap = np.empty((2000, len(levels)))
for i in range(len(bootstrap)):
    draw = rng.integers(0, len(groups), size=len(groups))
    bootstrap[i] = group_sums[draw].sum(axis=0) / group_sizes[draw].sum()

low, high = np.quantile(bootstrap, [0.025, 0.975], axis=0)
result = pd.DataFrame({
    "level": levels,
    "delta_MULTI_minus_EN": observed,
    "CI95_lower": low,
    "CI95_upper": high,
    "includes_zero": (low <= 0) & (high >= 0),
})
file_csv = run.parent / f"{run.name}_bootstrap_appaiato.csv"
result.to_csv(file_csv, index=False)
display(result.round(4))
print('MULTILINGUAL_BOOTSTRAP_OK — 399 documents, 395 groups, 2,000 resamples')
print('Saved results table:', file_csv)

,level,delta_MULTI_meno_EN,IC95_basso,IC95_alto,include_zero
0,coarse,-0.0145,-0.0535,0.0253,True
1,fine,-0.0081,-0.0443,0.0280,True


MULTILINGUAL_BOOTSTRAP_OK — 399 articoli, 395 gruppi, 2000 ricampionamenti
Tabella da scaricare: /kaggle/working/nlp_setup/runs/multilingual_SVM_EN_vs_MULTI_seed42_35e21ad56f_bootstrap_appaiato.csv


## 08 · Paired group bootstrap

Estimate 95% confidence intervals for the multilingual-minus-English sample-F1 difference using 2,000 paired resamples of leakage groups. The intervals include zero for both label granularities; these results do not establish a statistically reliable difference.

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd

run = Path(multilingual_training_result["run_directory"])
file_pred = run.parent / f"{run.name}_outer_evaluation.npz"
conditions = ["EN_only", "EN_plus_auxiliary"]

with np.load(file_pred, allow_pickle=False) as data:
    folds = data["fold"]
    labels = data["classes_fine"].astype(str)
    gold = data["gold_fine"].astype(bool)
    predictions = {c: data[f"{c}_fine"].astype(bool) for c in conditions}

assert labels.tolist() == list(taxonomy["fine"])
specific = labels != "Other"  # Exclude only the global Other category.
rows = []

for fold in (1, 2, 3):
    support = {}
    for condition in conditions:
        path = run / condition / f"fold_{fold}" / "training_label_support.npz"
        with np.load(path, allow_pickle=False) as data:
            support[condition] = data["fine"]

    en = support["EN_only"]
    multi = support["EN_plus_auxiliary"]
    rare = (en >= 1) & (en <= 4)
    bands = {
        "0 EN examples": en == 0,
        "1–4 EN examples": rare,
        "5–9 EN examples": (en >= 5) & (en <= 9),
        "10+ EN examples": en >= 10,
        "di cui: 1–4 EN e almeno 5 multilingui": rare & (multi >= 5),
    }
    for band, mask in bands.items():
        selected = mask & specific
        y = gold[folds == fold][:, selected]
        for condition in conditions:
            p = predictions[condition][folds == fold][:, selected]
            rows.append({
                "support_band": band, "condition": condition,
                "gold_annotations": int(y.sum()),
                "recuperate": int((p & y).sum()),
                "falsi_positivi": int((p & ~y).sum()),
            })

table = pd.DataFrame(rows).groupby(
    ["support_band", "condition"], sort=False, as_index=False
).sum()
table["recall"] = table["recuperate"] / table["gold_annotations"].replace(0, np.nan)
tot_pred = table["recuperate"] + table["falsi_positivi"]
table["precision"] = table["recuperate"] / tot_pred.replace(0, np.nan)

file_csv = run.parent / f"{run.name}_rare_labels.csv"
table.to_csv(file_csv, index=False)
display(table.round(4))
print('Saved results table:', file_csv)

,fascia,sistema,annotazioni_gold,recuperate,falsi_positivi,recall,precision
0,0 esempi EN,EN_only,19,0,0,0.0000,NaN
1,0 esempi EN,EN_plus_auxiliary,19,0,7,0.0000,0.0000
2,1–4 esempi EN,EN_only,210,6,19,0.0286,0.2400
3,1–4 esempi EN,EN_plus_auxiliary,210,30,106,0.1429,0.2206
4,5–9 esempi EN,EN_only,269,46,151,0.1710,0.2335
5,5–9 esempi EN,EN_plus_auxiliary,269,82,178,0.3048,0.3154
6,10+ esempi EN,EN_only,208,80,185,0.3846,0.3019
7,10+ esempi EN,EN_plus_auxiliary,208,97,205,0.4663,0.3212
8,di cui: 1–4 EN e almeno 5 multilingui,EN_only,191,5,17,0.0262,0.2273
9,di cui: 1–4 EN e almeno 5 multilingui,EN_plus_auxiliary,191,29,104,0.1518,0.2180


Tabella da scaricare: /kaggle/working/nlp_setup/runs/multilingual_SVM_EN_vs_MULTI_seed42_35e21ad56f_rare_labels.csv


## 09 · Rare-label diagnostics

Inspect coarse-grained support bands for fine labels, including the global `Other` class exclusion. Maintain the original diagnostics, including low-support outcomes.

In [11]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

run = Path(multilingual_training_result["run_directory"])
conditions = ["EN_only", "EN_plus_auxiliary"]
levels = ["coarse", "fine"]
fixed_threshold = MULTILINGUAL_CONFIG["fixed_threshold_comparator"]
assert fixed_threshold == 0.0

with np.load(run / "embedding_cache" / "embeddings.npz", allow_pickle=False) as cache:
    index = {key: i for i, key in enumerate(cache["keys"].astype(str))}
    features = cache["all_chunks"]

pred_path = run.parent / f"{run.name}_outer_evaluation.npz"
with np.load(pred_path, allow_pickle=False) as archive:
    outer = {name: archive[name] for name in archive.files}

keys = outer["keys"].astype(str)
folds = outer["fold"]
rows = []

for condition in conditions:
    for level in levels:
        gold = outer[f"gold_{level}"]
        previous = outer[f"{condition}_{level}"].astype(bool)
        fixed = np.zeros_like(previous)

        for fold in (1, 2, 3):
            mask = folds == fold
            bundle = joblib.load(run / condition / f"fold_{fold}" / "svm_models.joblib")
            x = features[[index[key] for key in keys[mask]]].astype(np.float64)
            scores = semtrain_scores(bundle["models"][level], x)

            reproduced = semtrain_decode(scores, bundle["thresholds"][level])
            assert np.array_equal(reproduced, previous[mask]), (
                f"Predictions differ from the saved reference: {condition}, {level}, fold {fold}"
            )
            fixed[mask] = semtrain_decode(scores, fixed_threshold)

        rows.append({
            "level": level,
            "condition": condition,
            "F1_validation_threshold": semtrain_f1(gold, previous),
            "F1_zero_threshold": semtrain_f1(gold, fixed),
            "documents_with_changed_predictions": int((fixed != previous).any(axis=1).sum()),
        })

table = pd.DataFrame(rows)
metrics = ["F1_validation_threshold", "F1_zero_threshold"]
by_condition = table.set_index(["condition", "level"])[metrics]
delta = by_condition.loc["EN_plus_auxiliary"] - by_condition.loc["EN_only"]

display(table.round(4))
display(delta.round(4).rename_axis("MULTI_meno_EN"))
csv_path = run.parent / f"{run.name}_fixed_threshold.csv"
table.to_csv(csv_path, index=False)
print('FIXED_THRESHOLD_OK — reproduced reference predictions on 399 documents')
print('Saved results table:', csv_path)

,level,condition,F1_soglia_validazione,F1_soglia_zero,articoli_con_predizione_cambiata
0,coarse,EN_only,0.5417,0.5217,141
1,fine,EN_only,0.4407,0.4344,126
2,coarse,EN_plus_auxiliary,0.5272,0.5285,167
3,fine,EN_plus_auxiliary,0.4326,0.4106,244


,F1_soglia_validazione,F1_soglia_zero
MULTI_meno_EN,,
coarse,-0.0145,0.0069
fine,-0.0081,-0.0238


FIXED_THRESHOLD_OK — predizioni precedenti riprodotte sui 399 articoli
Tabella da scaricare: /kaggle/working/nlp_setup/runs/multilingual_SVM_EN_vs_MULTI_seed42_35e21ad56f_fixed_threshold.csv


## 10 · Fixed-threshold sensitivity

Compare validation-selected thresholds with a fixed decision threshold of zero. This is a sensitivity analysis rather than an independent model comparison.

In [12]:
import json
from pathlib import Path

run = Path(multilingual_training_result["run_directory"])
config = json.loads((run / "config.json").read_text(encoding="utf-8"))

print('SVM parameters used in Notebook 07')
print(json.dumps(config["classifier"], indent=2, ensure_ascii=False))

Parametri della SVM usata nel notebook 07
{
  "C": 1.0,
  "penalty": "l2",
  "loss": "squared_hinge",
  "dual": "auto",
  "tol": 0.0001,
  "max_iter": 10000,
  "class_weight": null,
  "random_state": 42
}


## 11 · Classifier configuration

Display the saved SVM solver settings, including `random_state = 42` and the absence of class weighting.

In [13]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

run = Path(multilingual_training_result["run_directory"])
cfg = json.loads((run / "config.json").read_text())
pred_path = run.parent / f"{run.name}_outer_evaluation.npz"

with np.load(pred_path, allow_pickle=False) as data:
    outer = {name: data[name] for name in data.files}

with np.load(run / "embedding_cache" / "embeddings.npz", allow_pickle=False) as data:
    index = {key: i for i, key in enumerate(data["keys"].astype(str))}
    features = data["all_chunks"]

def features_for(keys):
    return features[[index[key] for key in keys]].astype(np.float64)

rows = []
for level in ("coarse", "fine"):
    gold = outer[f"gold_{level}"]
    baseline = outer[f"EN_only_{level}"].astype(bool)
    multi_f1 = semtrain_f1(gold, outer[f"EN_plus_auxiliary_{level}"])

    for seed in (42, 13, 87):
        predictions = baseline.copy()
        if seed != 42:
            for fold in (1, 2, 3):
                parts = setup.make_fold(train_records, manifest, similarity, fold, False)
                train = parts["inner_train"]
                val = parts["inner_validation"]
                mask = outer["fold"] == fold
                outer_keys = outer["keys"][mask].astype(str).tolist()
                assert set(outer_keys) == {r["key"] for r in parts["outer_evaluation"]}

                x_train = features_for([r["key"] for r in train])
                y_train = setup.binary_labels([r[level] for r in train], taxonomy[level])
                models = semtrain_fit(x_train, y_train, {
                    **cfg["classifier"], "random_state": seed
                })

                x_val = features_for([r["key"] for r in val])
                y_val = setup.binary_labels([r[level] for r in val], taxonomy[level])
                best, _ = semtrain_select(
                    y_val, semtrain_scores(models, x_val), cfg["threshold_grid"]
                )
                scores = semtrain_scores(models, features_for(outer_keys))
                predictions[mask] = semtrain_decode(scores, best["threshold"])

        en_f1 = semtrain_f1(gold, predictions)
        rows.append({
            "level": level,
            "seed_EN": seed,
            "F1_EN": en_f1,
            "F1_MULTI": multi_f1,
            "delta_MULTI_minus_EN": multi_f1 - en_f1,
            'EN_documents_changed_vs_seed42': int(
                (predictions != baseline).any(axis=1).sum()
            ),
        })

result = pd.DataFrame(rows)
csv_path = run.parent / f"{run.name}_solver_seeds.csv"
result.to_csv(csv_path, index=False)
display(result.round(6))
print('Saved results table:', csv_path)

,level,seed_EN,F1_EN,F1_MULTI,delta_MULTI_meno_EN,articoli_EN_cambiati_rispetto_a_42
0,coarse,42,0.541710,0.527165,-0.014545,0
1,coarse,13,0.541710,0.527165,-0.014545,0
2,coarse,87,0.541710,0.527165,-0.014545,0
3,fine,42,0.440689,0.432630,-0.008059,0
4,fine,13,0.440689,0.432630,-0.008059,0
5,fine,87,0.440689,0.432630,-0.008059,0


Tabella da scaricare: /kaggle/working/nlp_setup/runs/multilingual_SVM_EN_vs_MULTI_seed42_35e21ad56f_solver_seeds.csv
